# BRIGHTER PT-BR: A Reproducible Study of Multi-Label Emotion Classification

**Imbalance-aware learning, encoder selection, threshold calibration, ensemble modeling, and generalization analysis.**

This companion notebook tells the complete experimental story from frozen, machine-readable artifacts. It is designed to render directly on GitHub: the lightweight cells have already been executed and their outputs are embedded.

> **Reproducibility note**
>
> This notebook documents an experiment that has already been finalized. The official test set was evaluated once under a frozen protocol. All development, test, and post-hoc results displayed here are read from frozen artifacts. Running this notebook must not modify the official experimental results.

```text
state = TEST_EVALUATED
search_space = CLOSED
further_tuning = FORBIDDEN
```

In [1]:
import os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "reports/brighter_ptbr/paper/master_results.csv").exists():
            return candidate
    raise FileNotFoundError("Run from the project root, GitHub/, or a descendant of the project.")

PROJECT_ROOT = find_project_root(Path.cwd())
os.environ.setdefault("MPLCONFIGDIR", str((PROJECT_ROOT / ".cache" / "matplotlib").resolve()))
os.environ.setdefault("MPLBACKEND", "Agg")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RUN_FINAL_REPRODUCTION = False
ALLOW_TEST_ACCESS = False
assert RUN_FINAL_REPRODUCTION is False
assert ALLOW_TEST_ACCESS is False

LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
COLORS = {"development": "#4c78a8", "test": "#c44e52", "secondary": "#72b7b2"}
pd.set_option("display.precision", 4)

def read_json(relative_path):
    return json.loads((PROJECT_ROOT / relative_path).read_text(encoding="utf-8"))

print(f"Project root resolved: {PROJECT_ROOT.name}")
print(f"RUN_FINAL_REPRODUCTION={RUN_FINAL_REPRODUCTION}; ALLOW_TEST_ACCESS={ALLOW_TEST_ACCESS}")

# Frozen-artifact allowlist: no raw dataset, checkpoint, tokenizer, or model path is opened.
master = pd.read_csv(PROJECT_ROOT / "reports/brighter_ptbr/paper/master_results.csv")
evidence = read_json("metadata/brighter_ptbr/paper/paper_evidence_manifest.json")
eda = read_json("metadata/brighter_ptbr/eda/eda_summary.json")
baselines = read_json("metadata/brighter_ptbr/experiments/baseline_summary.json")
losses = read_json("metadata/brighter_ptbr/experiments/imbalance_loss_summary.json")
encoders = read_json("metadata/brighter_ptbr/experiments/encoder_summary.json")
thresholds = read_json("metadata/brighter_ptbr/experiments/threshold_calibration_summary.json")
ensemble = read_json("metadata/brighter_ptbr/experiments/ensemble_summary.json")
final = read_json("reports/brighter_ptbr/final/final_test_results.json")
generalization = read_json("metadata/brighter_ptbr/final/generalization_gap_summary.json")
freeze_manifest = read_json("metadata/brighter_ptbr/final/final_experiment_manifest.json")
state = read_json("metadata/brighter_ptbr/final/experiment_state.json")

assert state["state"] == "TEST_EVALUATED"
assert state["search_space"] == "CLOSED"
assert state["further_tuning_allowed"] is False
display(pd.DataFrame({
    "state": [state["state"]], "primary": [state["primary"]],
    "test_accessed": [state["test_accessed"]], "search_space": [state["search_space"]],
    "evidence_claims": [len(evidence["claims"])],
}))

Project root resolved: Artigo De Emocoes para Submissao Codex
RUN_FINAL_REPRODUCTION=False; ALLOW_TEST_ACCESS=False


,state,primary,test_accessed,search_space,evidence_claims
0,TEST_EVALUATED,EN2,True,CLOSED,369


## 1. Research problem

Each text may express **zero, one, or several** of six emotions: `anger`, `disgust`, `fear`, `joy`, `sadness`, and `surprise`. The output is therefore a six-dimensional binary vector rather than one softmax class.

Macro-F1 is the primary metric because it gives each emotion equal weight, but this also makes rare-label behavior central. For example, the synthetic sentence below is only pedagogical—it is **not** a BRIGHTER observation.

In [2]:
example = pd.DataFrame([{ "text": "Eu estava com medo, mas também muito irritado.",
    "anger": 1, "disgust": 0, "fear": 1, "joy": 0, "sadness": 0, "surprise": 0 }])
display(example)

,text,anger,disgust,fear,joy,sadness,surprise
0,"Eu estava com medo, mas também muito irritado.",1,0,1,0,0,0


## 2. Dataset

The study uses **BRIGHTER Track A, Brazilian Portuguese**, pinned to revision `419566ac8f46f951b51584be36c13e5363008144`. During development, official train and dev were combined into a 2,426-example pool so every candidate could be assessed with the same frozen cross-validation folds. The official test remained sealed until the one-time final evaluation.

In [3]:
split_table = pd.DataFrame([
    {"split": s, "examples": eda["splits"][s]["rows"], "role": "development" if s in {"train", "dev"} else "one-time final test"}
    for s in ["train", "dev", "test"]
])
split_table.loc[len(split_table)] = ["train + dev", 2426, "cross-validated development pool"]
display(split_table)

,split,examples,role
0,train,2226,development
1,dev,200,development
2,test,2226,one-time final test
3,train + dev,2426,cross-validated development pool


## 3. Data audit: selecting Track A correctly

The upstream Hugging Face `dev` and `test` payloads contained Track A and Track C rows together. The curated pipeline selected Track A through explicit IDs matching `ptbr_<split>_track_a_<number>`—not through `drop_duplicates`.

- Raw: train 2,226; dev 400; test 4,452.
- Track A: train 2,226; dev 200; test 2,226.
- 2,426 Track A/C duplicate groups and 0 A/C label conflicts were documented.
- One official train–test text overlap was known and retained under the frozen protocol.

This audit prevents accidental Track C contamination without changing the official Track A split.

In [4]:
audit = pd.DataFrame({
    "split": ["train", "dev", "test"],
    "raw rows": [2226, 400, 4452],
    "selected Track A": [2226, 200, 2226],
    "selection": ["explicit Track A IDs"] * 3,
})
display(audit)

,split,raw rows,selected Track A,selection
0,train,2226,2226,explicit Track A IDs
1,dev,400,200,explicit Track A IDs
2,test,4452,2226,explicit Track A IDs


## 4. Multi-label EDA and imbalance

The training data combine many neutral/all-zero examples with strong label imbalance. Only 32 of the 64 possible labelsets occur. MeanIR quantifies between-label imbalance; SCUMBLE summarizes concurrence between minority and majority labels.

In [5]:
train = eda["splits"]["train"]
eda_summary = pd.DataFrame({
    "statistic": ["examples", "zero-label", "single-label", "multi-label", "label cardinality", "label density", "observed labelsets", "MeanIR", "SCUMBLE"],
    "value": [2226, 632, 1261, 333, train["label_cardinality"], train["label_density"], "32 / 64", eda["mean_ir"], eda["scumble"]],
})
display(eda_summary)

label_counts = pd.Series({label: train["labels"][label]["positive"] for label in LABELS})
fig, ax = plt.subplots(figsize=(8, 3.7))
bars = ax.bar(label_counts.index, label_counts.values, color=["#c44e52" if x == "disgust" else "#4c78a8" for x in label_counts.index])
ax.bar_label(bars, padding=3); ax.set_ylabel("Positive training examples"); ax.set_title("Track A PT-BR label distribution")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); display(fig); plt.close(fig)

,statistic,value
0,examples,2226
1,zero-label,632
2,single-label,1261
3,multi-label,333
4,label cardinality,0.8796
5,label density,0.1466
6,observed labelsets,32 / 64
7,MeanIR,4.2198
8,SCUMBLE,0.0379


<Figure size 800x370 with 1 Axes>

### Why `disgust` matters

`disgust` has only 75 training positives. Sixty-five co-occur with at least one majority label and 60 co-occur with `anger`. Naive oversampling would therefore often replicate majority-label signal as well. This motivated prioritizing loss functions and leakage-safe thresholding before any data augmentation.

In [6]:
minority = pd.DataFrame(eda["imbalance"]["minority_concurrence"])
display(minority[["minority_label", "positive_examples", "with_majority_label", "most_frequent_co_label", "most_frequent_co_label_count"]])

,minority_label,positive_examples,with_majority_label,most_frequent_co_label,most_frequent_co_label_count
0,disgust,75,65,anger,60
1,fear,109,48,anger,28
2,surprise,153,83,anger,47


### Why `max_length = 128`?

With the frozen BERTimbau tokenizer audit, the 95th percentile was 76 tokens and the 99th percentile 144.75. Only 1.26% of training texts exceeded 128 tokens, compared with 0.27% exceeding 256. A length of 128 was selected as the fixed coverage–compute compromise.

In [7]:
lengths = eda["token_lengths"]
length_table = pd.DataFrame([
    {"quantity": "p95 tokens", "value": lengths["distribution"]["p95"]},
    {"quantity": "p99 tokens", "value": lengths["distribution"]["p99"]},
    {"quantity": "examples >128", "value": f"{lengths['truncation']['128']['truncated_percentage']:.2f}%"},
    {"quantity": "examples >256", "value": f"{lengths['truncation']['256']['truncated_percentage']:.2f}%"},
])
display(length_table)

,quantity,value
0,p95 tokens,76.0
1,p99 tokens,144.75
2,examples >128,1.26%
3,examples >256,0.27%


## 5. Frozen experimental protocol

All candidates used the same 2,426-example development pool and the same five iterative multilabel folds. Models trained on four folds produced out-of-fold (OOF) probabilities for the held-out fold; concatenating all held-out predictions yielded one prediction per example. The test set was not used for model, loss, encoder, threshold, or ensemble selection.

In [8]:
fig, ax = plt.subplots(figsize=(9, 2.5))
nodes = [(0.08, "Development\npool: 2,426"), (0.34, "Frozen 5-fold\nmultilabel CV"), (0.61, "One held-out\nfold at a time"), (0.87, "2,426 OOF\npredictions")]
for x, label in nodes:
    ax.text(x, .55, label, ha="center", va="center", bbox=dict(boxstyle="round,pad=.45", fc="#e9f1f7", ec="#42677f"))
for (x1, _), (x2, _) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", (x2-.10, .55), (x1+.10, .55), arrowprops=dict(arrowstyle="->", color="#555"))
ax.text(.5, .12, "same data • same folds • same metric definitions • test sealed", ha="center")
ax.axis("off"); fig.tight_layout(); display(fig); plt.close(fig)

<Figure size 900x250 with 1 Axes>

## 6. Baselines

The trajectory begins with a sparse linear baseline and then controlled BERTimbau fine-tuning. Greater encoder capacity improved aggregate representation, but B2 still produced `disgust` F1 = 0 at threshold 0.5. Its AP values—0.1550 for disgust, 0.3102 for fear, and 0.4528 for surprise—showed that zero F1 did not imply zero ranking signal.

In [9]:
baseline_view = master.query("experiment_id in ['B0','B1','B2']")[["experiment_id", "encoder", "loss", "macro_f1", "macro_f1_std"]].copy()
display(baseline_view)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
bars = ax.bar(baseline_view.experiment_id, baseline_view.macro_f1, color="#4c78a8")
ax.bar_label(bars, labels=[f"{x:.4f}" for x in baseline_view.macro_f1], padding=3)
ax.set_ylim(0, .46); ax.set_ylabel("Macro-F1 (fold mean)"); ax.set_title("Baseline progression")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); display(fig); plt.close(fig)

,experiment_id,encoder,loss,macro_f1,macro_f1_std
0,B0,TF-IDF word+char,Logistic Regression,0.1694,0.0120
1,B1,BERTimbau Base,BCE,0.3238,0.0204
2,B2,BERTimbau Large,BCE,0.3958,0.0212


<Figure size 650x340 with 1 Axes>

## 7. Imbalance-aware loss functions

- **BCE:** independent binary cross-entropy for each emotion, $-y\log p-(1-y)\log(1-p)$.
- **Weighted BCE:** multiplies positive terms by training-fold weights.
- **Focal Loss:** $(1-p_t)^\gamma\,\mathrm{BCE}$ downweights easy examples.
- **Weighted Focal:** combines class weights with focal modulation.
- **Asymmetric Loss (ASL):** uses different positive/negative focusing, here $\gamma_+=0$, $\gamma_-=4$, with negative clipping 0.05.

The hypothesis was that objectives emphasizing hard/rare positives could recover minority labels without changing the dataset.

In [10]:
loss_ids = ["B1", "L1", "L2", "L3", "L4"]
loss_view = master.set_index("experiment_id").loc[loss_ids, ["loss", "macro_f1", "neutral_fp", "disgust_f1"]].reset_index()
loss_view.loc[0, "loss"] = "BCE (Base control)"
display(loss_view)
fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.bar(loss_view.experiment_id, loss_view.macro_f1, color="#72b7b2")
ax.bar_label(bars, labels=[f"{x:.4f}" for x in loss_view.macro_f1], padding=3)
ax.set_ylim(0, .53); ax.set_ylabel("OOF Macro-F1"); ax.set_title("Loss-function comparison on BERTimbau Base")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); display(fig); plt.close(fig)

,experiment_id,loss,macro_f1,neutral_fp,disgust_f1
0,B1,BCE (Base control),0.3238,0.2847,0.0000
1,L1,Weighted BCE,0.4470,0.7684,0.1853
2,L2,Focal,0.3789,0.3097,0.0000
3,L3,Weighted Focal,0.4024,0.2552,0.1980
4,L4,ASL,0.4617,0.7729,0.2160


<Figure size 800x360 with 1 Axes>

**Decision.** Weighted BCE achieved strong Macro-F1 but aggressive neutral behavior. Weighted Focal preserved better minority precision/ranking (`disgust` F1 0.1980), while ASL achieved the best aggregate result (0.4617; `disgust` F1 0.2160) but overpredicted heavily. Weighted Focal and ASL advanced to the encoder study.

## 8. Encoder × loss comparison

BERTimbau is a general Portuguese BERT family; BERTweetBR was pretrained on Brazilian Portuguese tweets/social media; Albertina provides a different PT-BR encoder architecture at a comparable scale. The controlled matrix tests whether gains arise from the encoder alone or its interaction with the loss.

In [11]:
encoder_view = master.query("experiment_id in ['E1','E2','E3','E4','E5','E6']")[["experiment_id", "encoder", "loss", "macro_f1", "disgust_f1", "fear_f1", "surprise_f1"]]
display(encoder_view)
fig, ax = plt.subplots(figsize=(8.5, 3.6))
bars = ax.bar(encoder_view.experiment_id, encoder_view.macro_f1, color=["#4c78a8"]*3+["#f2cf5b"]*3)
ax.bar_label(bars, labels=[f"{x:.4f}" for x in encoder_view.macro_f1], padding=3)
ax.set_ylim(0, .60); ax.set_ylabel("OOF Macro-F1"); ax.set_title("Encoder × loss matrix")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); display(fig); plt.close(fig)

,experiment_id,encoder,loss,macro_f1,disgust_f1,fear_f1,surprise_f1
7,E1,BERTweetBR,BCE,0.3263,0.0000,0.0000,0.0000
8,E2,BERTweetBR,Weighted Focal,0.3974,0.1474,0.1527,0.2512
9,E3,BERTweetBR,ASL,0.5258,0.2632,0.3438,0.4706
10,E4,Albertina 100M PT-BR,BCE,0.2804,0.0000,0.0000,0.0000
11,E5,Albertina 100M PT-BR,Weighted Focal,0.2664,0.0220,0.0462,0.1852
12,E6,Albertina 100M PT-BR,ASL,0.3953,0.1608,0.1768,0.2933


<Figure size 850x360 with 1 Axes>

**Evidence and decision.** BERTweetBR+BCE (0.3263) remained close to BERTimbau Base+BCE (0.3238), so encoder replacement alone was not the explanation. The strongest fixed-threshold result was the interaction **BERTweetBR × ASL** (E3=0.5258). Albertina did not reproduce that gain.

## 9. Ranking is not the same as a binary decision

A fixed threshold of 0.5 can suppress a rare class even when relative scores rank positives above negatives. Scores such as 0.10, 0.18, and 0.20 never cross 0.5, yet their ordering can still contain useful information. Threshold calibration changes decisions; it does not create ranking signal.

In [12]:
concept_scores = np.array([.04, .07, .10, .18, .20, .24])
fig, ax = plt.subplots(figsize=(7.5, 2.7))
ax.scatter(range(len(concept_scores)), concept_scores, s=65, color="#4c78a8")
ax.axhline(.5, color="#c44e52", ls="--", label="global threshold 0.50")
ax.axhline(.15, color="#72b7b2", ls=":", label="illustrative lower threshold")
ax.set_ylim(0, .56); ax.set_ylabel("Illustrative disgust score"); ax.set_xlabel("Synthetic example rank")
ax.legend(frameon=False, ncol=2); ax.set_title("Ranking signal can exist below 0.5")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); display(fig); plt.close(fig)

<Figure size 750x270 with 1 Axes>

## 10. Leakage-safe threshold calibration

Thresholds are hyperparameters. Choosing them on the full OOF labels and reporting performance on those same examples would be optimistic. The project therefore used nested 5×3 calibration: each outer validation fold remained invisible while three-fold inner OOF predictions selected one threshold per class.

In [13]:
fig, ax = plt.subplots(figsize=(9, 2.6))
nodes = [(0.10, "Outer train"), (0.34, "3-fold inner CV\ninner OOF scores"), (0.59, "Select 6 thresholds\nthen freeze"), (0.86, "Outer validation\nuntouched labels")]
for x, label in nodes:
    ax.text(x, .56, label, ha="center", va="center", bbox=dict(boxstyle="round,pad=.5", fc="#eef5ea", ec="#557a46"))
for (x1, _), (x2, _) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", (x2-.12, .56), (x1+.12, .56), arrowprops=dict(arrowstyle="->", color="#555"))
ax.text(.5, .13, "Outer-validation labels never participate in threshold selection", ha="center", weight="bold")
ax.axis("off"); fig.tight_layout(); display(fig); plt.close(fig)

<Figure size 900x260 with 1 Axes>

## 11. Threshold results

Thresholding improved all three selected systems, including conventional BCE on BERTimbau Large. This demonstrates that imbalance-aware learning and decision calibration are complementary rather than interchangeable.

In [14]:
threshold_view = pd.DataFrame({
    "system": ["T1 Weighted Focal", "T2 BERTweetBR+ASL", "T3 Large+BCE"],
    "fixed 0.50": [0.3974, 0.5258, 0.3976],
    "nested thresholds": [0.5152, 0.5585, 0.5023],
})
threshold_view["delta"] = threshold_view["nested thresholds"] - threshold_view["fixed 0.50"]
display(threshold_view)
ax = threshold_view.set_index("system")[["fixed 0.50", "nested thresholds"]].plot.bar(figsize=(9, 3.8), color=["#b9c4cf", "#4c78a8"])
ax.set_ylabel("Macro-F1"); ax.set_ylim(0, .62); ax.set_title("Leakage-safe class-specific thresholding")
ax.tick_params(axis="x", rotation=0); ax.spines[["top", "right"]].set_visible(False)
fig = ax.figure; fig.tight_layout(); display(fig); plt.close(fig)

,system,fixed 0.50,nested thresholds,delta
0,T1 Weighted Focal,0.3974,0.5152,0.1178
1,T2 BERTweetBR+ASL,0.5258,0.5585,0.0327
2,T3 Large+BCE,0.3976,0.5023,0.1047


<Figure size 900x380 with 1 Axes>

## 12. Probability-level ensemble

T2 (BERTweetBR+ASL) was the strongest standalone system. T3 (BERTimbau Large+BCE) made sufficiently different errors to justify a leakage-safe soft ensemble:

$$p_{EN2}=\alpha p_{T2}+(1-\alpha)p_{T3}.$$

For every outer fold, both $\alpha$ and the six thresholds were selected only from paired, out-of-sample inner probabilities.

In [15]:
ensemble_view = master.set_index("experiment_id").loc[["T2", "T3", "EN1", "EN2"], ["decision_strategy", "macro_f1", "micro_f1", "neutral_fp", "predicted_cardinality"]].reset_index()
display(ensemble_view)
comparison = next(x for x in ensemble["bootstrap"] if x["comparison"] == "EN2_vs_T2")
display(pd.DataFrame([{
    "comparison": "EN2 − T2", "delta Macro-F1": comparison["delta"],
    "95% CI lower": comparison["ci_2_5"], "95% CI upper": comparison["ci_97_5"],
    "bootstrap samples": comparison["samples"],
}]))

,experiment_id,decision_strategy,macro_f1,micro_f1,neutral_fp,predicted_cardinality
0,T2,nested per-label thresholds,0.5585,0.6750,0.4425,1.1014
1,T3,nested per-label thresholds,0.5023,0.6248,0.4292,1.0746
2,EN1,equal-weight soft ensemble + nested thresholds,0.5548,0.6657,0.4971,1.2131
3,EN2,tuned-weight soft ensemble + nested thresholds,0.5691,0.6846,0.4159,1.0639


,comparison,delta Macro-F1,95% CI lower,95% CI upper,bootstrap samples
0,EN2 − T2,0.0106,0.0011,0.0202,10000


**Decision.** Equal weighting (EN1=0.5548) did not improve T2, but nested tuned weighting produced EN2=0.5691: a +0.0106 gain with percentile CI [+0.0011,+0.0202]. T3 was retained because its complementary information produced a small but practically relevant cross-fitted improvement.

## 13. Frozen final system

> ### FINAL PRIMARY SYSTEM
> **T2:** BERTweetBR + ASL  
> **T3:** BERTimbau Large + BCE  
> **Fusion:** 0.70 × T2 + 0.30 × T3  
> **Thresholds:** anger 0.67; disgust 0.36; fear 0.43; joy 0.65; sadness 0.58; surprise 0.49

The full-OOF deployment calibration score, 0.5897, is **not an unbiased performance estimate**. The leakage-safe development estimate remains 0.5691.

In [16]:
deployment = pd.DataFrame({
    "component": ["T2 weight", "T3 weight", *LABELS],
    "frozen value": [final["alpha_t2"], final["weight_t3"], *[final["en2_thresholds"][x] for x in LABELS]],
    "status": ["DEPLOYMENT_CALIBRATION_ONLY"] * 8,
})
display(deployment)
display(pd.DataFrame([
    {"quantity": "Leakage-safe nested EN2", "Macro-F1": final["development"]["EN2_nested_macro_f1"], "status": "LEAKAGE_SAFE_ESTIMATE"},
    {"quantity": "Full-OOF deployment calibration", "Macro-F1": final["development"]["EN2_deployment_calibration_macro_f1"], "status": "DEPLOYMENT_CALIBRATION_ONLY"},
]))

,component,frozen value,status
0,T2 weight,0.70,DEPLOYMENT_CALIBRATION_ONLY
1,T3 weight,0.30,DEPLOYMENT_CALIBRATION_ONLY
2,anger,0.67,DEPLOYMENT_CALIBRATION_ONLY
3,disgust,0.36,DEPLOYMENT_CALIBRATION_ONLY
4,fear,0.43,DEPLOYMENT_CALIBRATION_ONLY
5,joy,0.65,DEPLOYMENT_CALIBRATION_ONLY
6,sadness,0.58,DEPLOYMENT_CALIBRATION_ONLY
7,surprise,0.49,DEPLOYMENT_CALIBRATION_ONLY


,quantity,Macro-F1,status
0,Leakage-safe nested EN2,0.5691,LEAKAGE_SAFE_ESTIMATE
1,Full-OOF deployment calibration,0.5897,DEPLOYMENT_CALIBRATION_ONLY


## 14. One-time final test

The final test was evaluated once after the system, alpha, thresholds, label order, and reporting protocol were frozen. The result is reported unchanged—even though it is much lower than development.

In [17]:
final_comparison = pd.DataFrame({
    "system": ["EN2 (primary)", "T2 (secondary)"],
    "nested development": [final["development"]["EN2_nested_macro_f1"], final["development"]["T2_nested_macro_f1"]],
    "final test": [final["primary"]["metrics"]["macro_f1"], final["secondary"]["metrics"]["macro_f1"]],
})
final_comparison["difference"] = final_comparison["final test"] - final_comparison["nested development"]
display(final_comparison)
ax = final_comparison.set_index("system")[["nested development", "final test"]].plot.bar(figsize=(7.5, 3.8), color=[COLORS["development"], COLORS["test"]])
ax.set_ylabel("Macro-F1"); ax.set_ylim(0, .64); ax.set_title("Development versus one-time final test")
ax.tick_params(axis="x", rotation=0); ax.spines[["top", "right"]].set_visible(False)
fig = ax.figure; fig.tight_layout(); display(fig); plt.close(fig)

,system,nested development,final test,difference
0,EN2 (primary),0.5691,0.2852,-0.2839
1,T2 (secondary),0.5585,0.1569,-0.4016


<Figure size 750x380 with 1 Axes>

## 15. Final per-label behavior

EN2 retained strong precision but became extremely conservative for several emotions. `joy` and `surprise` remained comparatively robust; `disgust`, `fear`, and especially `sadness` suffered severe recall collapse.

In [18]:
en2_labels = pd.DataFrame(final["primary"]["metrics"]["per_label"]).T.reset_index(names="label")
display(en2_labels[["label", "precision", "recall", "f1", "support", "average_precision"]])
fig, ax = plt.subplots(figsize=(8, 3.7))
bars = ax.bar(en2_labels.label, en2_labels.f1, color="#c44e52")
ax.bar_label(bars, labels=[f"{x:.3f}" for x in en2_labels.f1], padding=3)
ax.set_ylim(0, .78); ax.set_ylabel("Final-test F1"); ax.set_title("EN2 final-test performance by emotion")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); display(fig); plt.close(fig)

,label,precision,recall,f1,support,average_precision
0,anger,0.7895,0.2575,0.3883,699.0,0.6987
1,disgust,0.3333,0.0141,0.0270,71.0,0.1288
2,fear,0.6111,0.0791,0.1401,139.0,0.2884
3,joy,0.8659,0.5586,0.6791,555.0,0.7909
4,sadness,0.7500,0.0175,0.0342,343.0,0.6695
5,surprise,0.4452,0.4395,0.4423,157.0,0.4489


<Figure size 800x370 with 1 Axes>

## 16. Generalization gap: labels stayed stable, decisions did not

Gold cardinality was almost unchanged: 0.882111 in development and 0.882300 in test. EN2 predicted cardinality, however, fell from 1.0639 to 0.3459. The model produced far fewer positive decisions even though the average number of gold labels stayed stable.

In [19]:
cardinality = pd.DataFrame({
    "quantity": ["Gold cardinality", "EN2 predicted cardinality", "T2 predicted cardinality"],
    "development": [0.882111, 1.0639, 1.1014],
    "test": [0.882300, 0.3459, 0.3616],
})
display(cardinality)
ax = cardinality.set_index("quantity").plot.bar(figsize=(8.5, 3.8), color=[COLORS["development"], COLORS["test"]])
ax.axhline(.8823, color="#555", ls=":", lw=1); ax.set_ylabel("Labels per example"); ax.set_title("Cardinality shift")
ax.tick_params(axis="x", rotation=0); ax.spines[["top", "right"]].set_visible(False)
fig = ax.figure; fig.tight_layout(); display(fig); plt.close(fig)

,quantity,development,test
0,Gold cardinality,0.8821,0.8823
1,EN2 predicted cardinality,1.0639,0.3459
2,T2 predicted cardinality,1.1014,0.3616


<Figure size 850x380 with 1 Axes>

### Label prevalence does not explain the gap

The development and test gold prevalences are close. `fear` rises modestly and `joy` decreases modestly, but there is no broad label-prevalence change commensurate with the Macro-F1 collapse.

In [20]:
prevalence = pd.read_csv(PROJECT_ROOT / "reports/brighter_ptbr/final/generalization_prevalence.csv")
display(prevalence[["label", "gold_dev", "gold_test", "gold_absolute_delta"]])
ax = prevalence.set_index("label")[["gold_dev", "gold_test"]].plot.bar(figsize=(8.5, 3.6), color=[COLORS["development"], COLORS["test"]])
ax.set_ylabel("Gold prevalence"); ax.set_title("Gold label prevalence is broadly stable")
ax.tick_params(axis="x", rotation=0); ax.spines[["top", "right"]].set_visible(False)
fig = ax.figure; fig.tight_layout(); display(fig); plt.close(fig)

,label,gold_dev,gold_test,gold_absolute_delta
0,anger,0.3199,0.3140,-0.0059
1,disgust,0.0330,0.0319,-0.0011
2,fear,0.0482,0.0624,0.0142
3,joy,0.2650,0.2493,-0.0157
4,sadness,0.1443,0.1541,0.0098
5,surprise,0.0717,0.0705,-0.0012


<Figure size 850x360 with 1 Axes>

### Frozen-threshold crossing rates collapsed

The same EN2 thresholds were applied in development and test. Crossing rates fell sharply for anger, disgust, fear, joy, and sadness; only surprise increased. This directly explains much of the lower predicted cardinality and recall, without implying that threshold transfer is the sole cause.

In [21]:
crossing = pd.read_csv(PROJECT_ROOT / "reports/brighter_ptbr/final/generalization_threshold_crossing.csv").query("system == 'EN2'")
display(crossing[["label", "threshold", "development_crossing_rate", "test_crossing_rate", "delta"]])
ax = crossing.set_index("label")[["development_crossing_rate", "test_crossing_rate"]].plot.bar(figsize=(8.5, 3.7), color=[COLORS["development"], COLORS["test"]])
ax.set_ylabel("Fraction above frozen threshold"); ax.set_title("EN2 threshold-crossing shift")
ax.tick_params(axis="x", rotation=0); ax.spines[["top", "right"]].set_visible(False)
fig = ax.figure; fig.tight_layout(); display(fig); plt.close(fig)

,label,threshold,development_crossing_rate,test_crossing_rate,delta
6,anger,0.67,0.3042,0.1024,-0.2018
7,disgust,0.36,0.1105,0.0013,-0.1091
8,fear,0.43,0.0515,0.0081,-0.0434
9,joy,0.65,0.2601,0.1608,-0.0993
10,sadness,0.58,0.1575,0.0036,-0.1539
11,surprise,0.49,0.0581,0.0696,0.0115


<Figure size 850x370 with 1 Axes>

## 17. Ranking degradation also occurred

EN2 Macro Average Precision fell from 0.6002 to 0.5042. Therefore the gap was not only a hard-threshold problem: ranking quality also deteriorated. These AP results are **post-hoc diagnostic metrics**, never used to change the frozen system.

In [22]:
calibration = pd.DataFrame(generalization["macro_calibration"])
en2_ap = calibration.query("system == 'EN2'")[["split", "average_precision", "brier_score"]]
display(en2_ap)
fig, ax = plt.subplots(figsize=(5.8, 3.3))
bars = ax.bar(en2_ap["split"], en2_ap["average_precision"], color=[COLORS["development"], COLORS["test"]])
ax.bar_label(bars, labels=[f"{x:.4f}" for x in en2_ap.average_precision], padding=3)
ax.set_ylim(0, .68); ax.set_ylabel("Macro Average Precision"); ax.set_title("Threshold-free ranking diagnostic")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); display(fig); plt.close(fig)

,split,average_precision,brier_score
0,development,0.6002,0.1275
1,test,0.5042,0.1215


<Figure size 580x330 with 1 Axes>

> ### Main finding
>
> The evidence is consistent with a combination of (1) probability-scale/decision-transfer shift between OOF fold models and final retrained models, (2) ranking degradation on the final test, and (3) greater instability of T2. T3 was comparatively more stable and the ensemble mitigated—but did not eliminate—the failure. Label-prevalence shift is not supported as the primary explanation; textual/domain shift remains inconclusive. No causal mechanism is claimed.

## 18. Lessons learned

1. Macro-F1 can conceal very different precision/recall regimes.
2. Rare-label AP can expose useful signal even when F1@0.5 is zero.
3. Encoder and loss interact; their effects are not simply additive.
4. Thresholds must be calibrated without leakage.
5. OOF-calibrated decision rules may not transfer reliably to final retrained models.
6. One untouched final test remains essential—even after rigorous nested development evaluation.

## 19. Computational and reproduction profile

The final ensemble requires two operational networks: approximately 135M BERTweetBR parameters plus 334M BERTimbau Large parameters, or about 469M parameters across both networks. This extra cost was accepted only after the leakage-safe ensemble comparison.

In [23]:
repro_config = pd.DataFrame([
    {"setting": "seed", "T2": 42, "T3": 42},
    {"setting": "max_length", "T2": 128, "T3": 128},
    {"setting": "epochs", "T2": 4, "T3": 4},
    {"setting": "learning_rate", "T2": "2e-5", "T3": "2e-5"},
    {"setting": "optimizer", "T2": "AdamW", "T3": "AdamW"},
    {"setting": "weight_decay", "T2": .01, "T3": .01},
    {"setting": "warmup_ratio", "T2": .1, "T3": .1},
    {"setting": "scheduler", "T2": "linear", "T3": "linear"},
    {"setting": "effective_batch", "T2": 16, "T3": 16},
    {"setting": "physical_batch / accumulation", "T2": "frozen E3 config", "T3": "1 / 16"},
])
display(repro_config)
display(pd.DataFrame([
    {"network": "BERTimbau Base", "parameters (approx.)": 109_000_000},
    {"network": "BERTweetBR", "parameters (approx.)": 135_000_000},
    {"network": "BERTimbau Large", "parameters (approx.)": 334_000_000},
    {"network": "Final ensemble (two networks)", "parameters (approx.)": 469_000_000},
]))
print(f"Final inference: T2={final['inference']['T2']['seconds']:.2f}s; T3={final['inference']['T3']['seconds']:.2f}s on 2,226 examples.")

,setting,T2,T3
0,seed,42,42
1,max_length,128,128
2,epochs,4,4
3,learning_rate,2e-5,2e-5
4,optimizer,AdamW,AdamW
5,weight_decay,0.01,0.01
6,warmup_ratio,0.1,0.1
7,scheduler,linear,linear
8,effective_batch,16,16
9,physical_batch / accumulation,frozen E3 config,1 / 16


,network,parameters (approx.)
0,BERTimbau Base,109000000
1,BERTweetBR,135000000
2,BERTimbau Large,334000000
3,Final ensemble (two networks),469000000


Final inference: T2=4.65s; T3=51.80s on 2,226 examples.


## 20. Optional: reproducing the final training

This guarded section is **not part of the official experiment**. It never accesses or evaluates the test set and must never overwrite frozen models. If a user explicitly changes `RUN_FINAL_REPRODUCTION` to `True`, the existing training implementation is redirected to `GitHub/reproduction_artifacts/`.

The notebook is distributed with `RUN_FINAL_REPRODUCTION = False` and `ALLOW_TEST_ACCESS = False`; therefore the cell below only defines a wrapper and performs no model import, loading, or training.

In [24]:
def reproduce_final_component(system: str):
    # Explicit opt-in reproduction; output is isolated from official frozen artifacts.
    if not RUN_FINAL_REPRODUCTION:
        raise RuntimeError("Set RUN_FINAL_REPRODUCTION=True manually to request a non-official reproduction.")
    if ALLOW_TEST_ACCESS:
        raise RuntimeError("Test access is forbidden in this notebook.")
    if system not in {"T2", "T3"}:
        raise ValueError("system must be T2 or T3")

    import sys
    from unittest.mock import patch
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    import brighter.finalization as finalization

    names = {"T2": "t2_bertweetbr_asl", "T3": "t3_bertimbau_large_bce"}
    safe_root = (PROJECT_ROOT / "GitHub" / "reproduction_artifacts").resolve()
    target = (safe_root / names[system]).resolve()
    if not target.is_relative_to(safe_root):
        raise RuntimeError("Unsafe reproduction output path")

    # train_final_model resolves this directory below artifacts/models/final;
    # three '..' segments return to PROJECT_ROOT before entering GitHub/.
    redirected = f"../../../GitHub/reproduction_artifacts/{names[system]}"
    patched_spec = {**finalization.FINAL_SPECS[system], "directory": redirected}
    with patch.dict(finalization.FINAL_SPECS, {system: patched_spec}, clear=False):
        return finalization.train_final_model(PROJECT_ROOT, system)

print("Optional reproduction guard active: no model loaded and no training executed.")

Optional reproduction guard active: no model loaded and no training executed.


## 21. References

- **BRIGHTER / SemEval-2025 Task 11:** multilingual multi-label emotion detection dataset and shared task.
- **BERTimbau:** Portuguese BERT models used as Base and Large controls.
- **BERTweetBR:** Brazilian Portuguese social-media encoder.
- **Focal Loss:** imbalance-aware objective that downweights easy examples.
- **Asymmetric Loss:** positive/negative asymmetric focusing for multi-label classification.

The repository's paper evidence package tracks which bibliographic entries still require final metadata verification; this notebook introduces no new literature claims.

## 22. Repository structure

- `reports/brighter_ptbr/paper/` contains the manuscript-ready evidence package.
- `metadata/brighter_ptbr/` contains machine-readable provenance and frozen experiment state.
- `reports/brighter_ptbr/experiments/` contains stage-specific scientific reports.
- `artifacts/` retains the official models and predictions; this notebook never modifies them.
- `GitHub/BRIGHTER_PTBR_Experiment.ipynb` is the concise reader-facing interface.

The official result remains **EN2 final-test Macro-F1 = 0.2852**. The experiment state remains `TEST_EVALUATED`, and further tuning is forbidden.

In [25]:
integrity_summary = pd.DataFrame([{
    "artifact mode": "frozen read-only",
    "models loaded": "NO",
    "training executed": "NO",
    "inference executed": "NO",
    "raw test loaded": "NO",
    "test rerun": "NO",
    "RUN_FINAL_REPRODUCTION": RUN_FINAL_REPRODUCTION,
    "ALLOW_TEST_ACCESS": ALLOW_TEST_ACCESS,
    "state": state["state"],
    "search space": state["search_space"],
}])
display(integrity_summary)

,artifact mode,models loaded,training executed,inference executed,raw test loaded,test rerun,RUN_FINAL_REPRODUCTION,ALLOW_TEST_ACCESS,state,search space
0,frozen read-only,NO,NO,NO,NO,NO,False,False,TEST_EVALUATED,CLOSED
